# R1.4 — адресные регрессии ядра
Не основной экран автора и не полная статистическая карта. Все числа синтетические. Проверяем время, общие входы и единый расчётный путь; не запускаем обучение/PDF.

In [ ]:
from pathlib import Path
import os, sys, json
root=Path.cwd()
while root != root.parent and not (root/'efc/model.py').exists():
    root=root.parent
assert (root/'efc/model.py').exists(), 'Сначала получите рабочую ветку репозитория'
os.chdir(root)
sys.path.insert(0,str(root))
from efc.model import Config, simulate, COMPONENTS
print(sys.version)

Граница 12 мс: равный приход принимается; на 1 нс раньше теряется при B=0; позже принимается. Проверка целых тиков, без допуска.

In [ ]:
bc=Config(n_edge=1,n_fog=1,n_cloud=1,buffer=0,n_tasks=2,mobile_fraction=0,edge_fixed_ms=(3,3),edge_processing_share=1/3,fog_service_ms=(9,9),fog_cloud_ms=(0,0),cloud_service_ms=(0,0))
boundaries=[]
for name,arrival,expected in [('equal','0.009',(2,0)),('early','0.008999999',(1,1)),('late','0.009000001',(2,0))]:
    br=simulate(bc,['0',arrival])
    got=(br['metrics']['completed'],br['metrics']['dropped'])
    assert got==expected,(name,got)
    boundaries.append({'case':name,'completed':got[0],'dropped':got[1],'first_fog_end_tick':br['tasks'][0]['fog_end_tick'],'second_fog_arrival_tick':br['tasks'][1]['fog_arrival_tick']})
out=root/'results/r14/current';out.mkdir(parents=True,exist_ok=True)
(out/'clock-boundaries.json').write_text(json.dumps(boundaries,indent=2)+chr(10))
print(json.dumps(boundaries,indent=2))

Один Fog, B=2. Ручные ответы заданы литералами до запуска ядра.

In [ ]:
c=Config(n_edge=1,n_fog=1,n_cloud=1,buffer=2,n_tasks=5,mobile_fraction=0,edge_fixed_ms=(2,2),edge_processing_share=.5,fog_service_ms=(10,10),fog_cloud_ms=(1,1),cloud_service_ms=(1,1))
r=simulate(c,[0,.001,.002,.003,.010])
expected_wait=[0,9,18,None,20]
expected_latency=[14,23,32,None,34]
for i,row in enumerate(r['tasks']):
    if i==3:
        assert row['status']=='dropped' and 'latency_ms' not in row
    else:
        assert abs(row['fog_wait_ms']-expected_wait[i])<1e-9
        assert abs(row['latency_ms']-expected_latency[i])<1e-9
        assert abs(sum(row[k] for k in COMPONENTS)-row['latency_ms'])<1e-9
print('Ручные ответы совпали:',expected_wait,expected_latency)

Модельные интервалы Fog сопоставлены с ручными. Потерянная задача не получает интервал обслуживания.

In [ ]:
import matplotlib.pyplot as plt
fig,ax=plt.subplots(figsize=(8,3))
manual={0:(2,12),1:(12,22),2:(22,32),4:(32,42)}
for row in r['tasks']:
    i=row['task_id']
    if i in manual:
        a,b=manual[i]
        ax.barh(i,(row['fog_end_s']-row['fog_start_s'])*1000,left=row['fog_start_s']*1000,height=.32,label='Модель' if i==0 else None)
        ax.plot([a,b],[i+.25,i+.25],'--',label='Ручной ответ' if i==0 else None)
    else:
        ax.scatter([row['drop_s']*1000],[i],marker='x')
ax.set_yticks(range(5),[str(i) for i in range(1,6)])
ax.set_xlabel('Время, мс'); ax.set_ylabel('Задача'); ax.legend(); fig.tight_layout()
from IPython.display import display, Image
image=root/'results/r14/current/manual-five.png'
image.parent.mkdir(parents=True,exist_ok=True)
fig.savefig(image,dpi=140)
plt.close(fig)
display(Image(filename=str(image)))

In [ ]:
out=root/'results/r14/current';out.mkdir(parents=True,exist_ok=True)
(out/'manual-five.json').write_text(json.dumps(r,ensure_ascii=False,indent=2)+chr(10))
assert r['metrics']['completed']==4 and r['metrics']['dropped']==1
print('OK: 4 завершения, 1 потеря. Это не проверка статьи/карты. Нейросеть и PDF не запускались.')

Парные входы: один короткий синтетический поток, меняются только Fog/B. Проверяем все исходные поля, включая потенциальные длительности потерянных сообщений. Это регрессия, не серия с прогревом и ДИ.

In [ ]:
from dataclasses import replace
from efc.workload import generate_workload, INPUT_FIELDS
from efc.stationary import stationary_run, observation_window
pc=Config(n_cloud=40,n_tasks=120,arrival_rate_s=.95*50/.055,seed=7)
w=generate_workload(pc)
sources=[e.record() for e in w.events]
checks=[]
for f in (30,40,50,60,70):
    for B in (0,2,5,10,20,50,100,200,400):
        cfg=replace(pc,n_fog=f,buffer=B)
        independently_generated=generate_workload(cfg)
        assert independently_generated.events==w.events
        rr=simulate(cfg,workload=w)
        assert [{k:t[k] for k in INPUT_FIELDS} for t in rr['tasks']]==sources
        assert all(t['fog']==t['sensor']%f for t in rr['tasks'])
        checks.append({'fog':f,'buffer':B,'input_sha256':rr['workload']['sha256'],
                       'completed':rr['metrics']['completed'],'dropped':rr['metrics']['dropped'],
                       'dropped_event_ids':[t['event_id'] for t in rr['tasks'] if t['status']=='dropped'],
                       'all_source_fields_equal':True})
assert any(x['dropped'] for x in checks)
assert len({x['input_sha256'] for x in checks})==1
(out/'paired-inputs.json').write_text(json.dumps({'scope':'small regression, NOT scientific grid','source':w.metadata(),'events':sources},indent=2)+chr(10))
(out/'paired-configurations.json').write_text(json.dumps(checks,indent=2)+chr(10))
print('45 вариантов, 120 одинаковых исходных сообщений каждый; потерянные включены в проверку.')

Оконный расчёт вызывает то же ядро. Ручные поступления сдвинуты на 1 мс в измерительное окно. Сравниваются полные записи и очередь, затем ручные ответы. Одна реализация не подтверждает допустимость конфигурации.

In [ ]:
wc=replace(c,arrival_rate_s=500)
window=observation_window(wc,.01,.2,1)
ww=replace(generate_workload(wc,['.001','.002','.003','.004','.011']),horizon_tick=window[2])
direct=simulate(wc,workload=ww)
row,wrapped=stationary_run(wc,.01,.2,1,workload=ww,return_details=True)
assert direct==wrapped
assert [t.get('latency_ms') for t in wrapped['tasks']]==[14,23,32,None,34]
assert (row['arrivals'],row['completed'],row['dropped'])==(5,4,1)
assert row['H200']==.8 and row['drop_fraction']==.2
assert row['statistical_acceptance']=='NOT_EVALUATED_SINGLE_REPLICATION'
(out/'window-path.json').write_text(json.dumps({'all_kernel_records_equal':True,'row':row,'result':wrapped},indent=2)+chr(10))
print('Единый путь подтверждён; 5 поступлений, 4 завершения, 1 потеря. Карта/статья не проверялись.')